<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 7 · Portfolio Risk and FX Conversion

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/07_dxquant_portfolio_risk_and_fx.ipynb)


## Notebook Goals

This notebook builds a small Monte Carlo portfolio and shows how to:

- aggregate discounted payoff distributions across positions,
- compute distribution-based risk measures (VaR, expected shortfall),
- convert multi-currency positions into a base currency, and
- run simple scenario shocks.

The portfolio examples here focus on mechanics. Correlated multi-risk portfolios of the
legacy `dx` style are out of scope for this notebook.

Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## 1. Shared Pricing Setup

To aggregate discounted payoff distributions across positions, we price them on the same
simulation specification (same model, paths, and time grid).

In [ ]:
pricing_date = dt.date(2024, 1, 1)
maturity = dt.date(2025, 1, 1)

spot = 100.0
rate = 0.02
vol = 0.20

env = dx.MarketEnvironment(name="port", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, rate))

process = dx.GeometricBrownianMotion(drift=rate, volatility=vol, seed=123)
pricer = dx.MonteCarloPricer(
    env=env,
    process=process,
    discount_curve_key="discount",
    paths=40000,
    steps=96,
)

pricer

## 2. Positions (Two Currencies)

We create two positions with different payoff types and currencies. FX conversion is applied
at the discounted-payoff level.

In [ ]:
call_usd = dx.EuropeanCallPayoff(strike=105.0)
barrier_eur = dx.UpAndOutBarrierCallPayoff(strike=100.0, barrier=130.0)

specs = [
    dx.PositionSpec(
        position_id="call_usd",
        pricer=pricer,
        payoff=call_usd,
        maturity=maturity,
        scenario_set="shared-gbm-seed-123",
        spot=spot,
        quantity=3.0,
        currency="USD",
    ),
    dx.PositionSpec(
        position_id="barrier_eur",
        pricer=pricer,
        payoff=barrier_eur,
        maturity=maturity,
        scenario_set="shared-gbm-seed-123",
        spot=spot,
        quantity=2.0,
        path_dependent=True,
        currency="EUR",
    ),
]

base_currency = "USD"
fx_rates = {"EUR": 1.10}

## 3. Portfolio Valuation and Risk Measures

VaR and expected shortfall are computed from the discounted payoff distribution. The sign
convention used here treats `price - payoff` as a loss.

In [ ]:
portfolio_pricer = dx.PortfolioPricer()
port = portfolio_pricer.price(
    specs,
    base_currency=base_currency,
    fx_rates=fx_rates,
)

print(f"Portfolio price:     {port.price:.6f} {base_currency}")
print(f"VaR 95%:             {port.value_at_risk(0.95):.6f} {base_currency}")
print(
    f"ES 95%:              {port.expected_shortfall(0.95):.6f} {base_currency}"
)

for pr in port.position_results:
    msg = f"{pr.spec.position_id:>11s}: {pr.price: .6f} {base_currency}"
    print(msg)

## 4. Scenario Shocks

Scenarios modify selected position specifications and reprice the portfolio.

In [ ]:
analyzer = dx.ScenarioAnalyzer(portfolio_pricer, specs)

scenarios = [
    dx.Scenario(name="base", modifier=lambda s: s),
    dx.Scenario(name="call +5%", modifier=dx.scale_spot("call_usd", 1.05)),
    dx.Scenario(name="call -5%", modifier=dx.scale_spot("call_usd", 0.95)),
]

results = analyzer.run(
    scenarios,
    base_currency=base_currency,
    fx_rates=fx_rates,
)

for res in results:
    print(f"{res.name:>9s}: {res.result.price: .6f} {base_currency}")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">